# Resolução dos Exercícios — Data Warehouse COVID-19 (ES)
**FAESA – Ciência da Computação | Business Intelligence 2026/1**  
Prof. Otávio Lube

---
## Exercício 1 — Análise Exploratória com pandas

O script abaixo lê o arquivo em chunks para evitar estouro de memória, acumula a contagem de nulos e valores vazios por coluna, e imprime um relatório ordenado por percentual de nulos.

In [ ]:
import pandas as pd

# Leitura em chunks para não estourar memória
chunks = pd.read_csv(
    "MICRODADOS.csv",
    sep=";",
    encoding="latin-1",
    chunksize=100_000
)

total_linhas = 0
nulos_por_col = None

for chunk in chunks:
    total_linhas += len(chunk)
    nulos = chunk.isnull().sum() + (chunk == "").sum()
    nulos_por_col = nulos if nulos_por_col is None else nulos_por_col + nulos

relatorio = pd.DataFrame({
    "nulos": nulos_por_col,
    "pct_nulo": (nulos_por_col / total_linhas * 100).round(2)
}).sort_values("pct_nulo", ascending=False)

print(f"Total de linhas: {total_linhas:,}")
print(relatorio.to_string())

**Interpretação esperada:**  
Colunas como `ResultadoRT_PCR` devem apresentar ~78% de nulos/não informados, confirmando o que a AED do tutorial identificou. Isso justifica o tratamento `COALESCE(..., 'Desconhecido')` no ETL, garantindo que valores ausentes virem membros reais das dimensões com SK = -1.

---
## Exercício 2 — Modelo Floco de Neve vs. Estrela para DIM_LOCALIDADE

### Proposta de modelo snowflake

In [ ]:
# DDL – Modelo Snowflake para localidade
sql_snowflake = """
-- Tabela de municípios normalizada
CREATE TABLE dw.dim_municipio (
    sk_municipio SERIAL PRIMARY KEY,
    municipio    VARCHAR(100),
    regiao_es    VARCHAR(30),
    macrorregiao VARCHAR(30),
    uf           CHAR(2) DEFAULT 'ES'
);

-- Tabela de bairros referenciando município
CREATE TABLE dw.dim_bairro (
    sk_bairro    SERIAL PRIMARY KEY,
    bairro       VARCHAR(150),
    sk_municipio INT REFERENCES dw.dim_municipio(sk_municipio)
);
"""
print(sql_snowflake)

### Comparação: Estrela vs. Floco de Neve

| Critério | Estrela | Floco de Neve |
|---|---|---|
| Número de joins nas queries | 1 | 2 ou mais |
| Performance OLAP | ✅ Melhor | ❌ Mais lento |
| Espaço em disco | Ligeira redundância | Menor (normalizado) |
| Manutenção | ✅ Mais simples | Mais complexa |
| Atualização de atributos | Linha única na dimensão | Tabelas separadas |
| Recomendação Kimball | ✅ Preferido | Evitar salvo necessidade |

**Conclusão:** Para OLAP, o modelo estrela é preferível. O floco de neve só se justificaria se `regiao_es` ou `macrorregiao` fossem atualizados frequentemente e de forma independente do município.

---
## Exercício 3 — Nova Tabela Fato: `fato_exame`

**Grão:** um exame realizado (uma linha por tipo de coleta com resultado registrado).  
As dimensões `DIM_TEMPO`, `DIM_LOCALIDADE` e `DIM_PERFIL_PACIENTE` são **conformed dimensions**: compartilhadas com `fato_notificacao_covid`, permitindo drill-across entre as duas tabelas fato.

In [ ]:
sql_fato_exame = """
CREATE TABLE dw.fato_exame (
    sk_fato_exame      BIGSERIAL PRIMARY KEY,

    -- Dimensões conformadas (compartilhadas com fato_notificacao)
    sk_data_coleta     INT NOT NULL REFERENCES dw.dim_tempo(sk_tempo),
    sk_data_resultado  INT NOT NULL REFERENCES dw.dim_tempo(sk_tempo),
    sk_local           INT NOT NULL REFERENCES dw.dim_localidade(sk_local),
    sk_perfil          INT NOT NULL REFERENCES dw.dim_perfil_paciente(sk_perfil),

    -- Dimensão específica de exame
    sk_teste           INT NOT NULL REFERENCES dw.dim_teste(sk_teste),

    -- Medidas
    qtd_exames         SMALLINT NOT NULL DEFAULT 1,
    flag_positivo      SMALLINT NOT NULL DEFAULT 0, -- 1 se resultado positivo
    dias_coleta_result INT                           -- latência coleta→resultado
);
"""
print(sql_fato_exame)

---
## Exercício 4 — Data Mart com Materialized View

In [ ]:
sql_mv = """
-- Criação da materialized view no schema mart
CREATE MATERIALIZED VIEW mart.mv_resumo_municipio_mes AS
SELECT
    l.municipio,
    t.ano_mes,
    SUM(f.flag_confirmado)   AS confirmados,
    SUM(f.flag_obito_covid)  AS obitos,
    SUM(f.flag_internado)    AS internacoes,
    SUM(f.qtd_notificacao)   AS notificacoes
FROM dw.fato_notificacao_covid f
JOIN dw.dim_localidade l ON l.sk_local = f.sk_local
JOIN dw.dim_tempo      t ON t.sk_tempo = f.sk_data_notificacao
GROUP BY l.municipio, t.ano_mes
WITH DATA;

-- Índice para acelerar filtros
CREATE INDEX idx_mv_municipio_mes
    ON mart.mv_resumo_municipio_mes(municipio, ano_mes);

-- Atualizar após cada carga ETL
REFRESH MATERIALIZED VIEW mart.mv_resumo_municipio_mes;
"""
print(sql_mv)

In [ ]:
sql_explain = """
-- Sem materialized view (query na fato com 5M linhas)
EXPLAIN ANALYZE
SELECT l.municipio, t.ano_mes, SUM(f.flag_confirmado)
FROM dw.fato_notificacao_covid f
JOIN dw.dim_localidade l ON l.sk_local = f.sk_local
JOIN dw.dim_tempo t ON t.sk_tempo = f.sk_data_notificacao
GROUP BY l.municipio, t.ano_mes;

-- Com materialized view (poucos milhares de linhas)
EXPLAIN ANALYZE
SELECT municipio, ano_mes, confirmados
FROM mart.mv_resumo_municipio_mes;
"""
print(sql_explain)

**Ganho esperado:** de segundos para milissegundos, pois a agregação é pré-computada e o resultado cabe inteiramente em memória.

---
## Exercício 5 — Dashboard (Power BI / Metabase)

**Conexão:** no Metabase ou Power BI, conectar ao banco `dw_covid` em `localhost:5432` com usuário `postgres`.

### (a) Série temporal de notificações

In [ ]:
sql_serie_temporal = """
SELECT t.ano_mes, SUM(f.qtd_notificacao) AS notificacoes
FROM dw.fato_notificacao_covid f
JOIN dw.dim_tempo t ON t.sk_tempo = f.sk_data_notificacao
GROUP BY t.ano_mes
ORDER BY t.ano_mes;
"""
print(sql_serie_temporal)

### (b) Mapa de calor por município

In [ ]:
sql_mapa = """
SELECT l.municipio, SUM(f.flag_confirmado) AS confirmados
FROM dw.fato_notificacao_covid f
JOIN dw.dim_localidade l ON l.sk_local = f.sk_local
GROUP BY l.municipio
ORDER BY confirmados DESC;
"""
print(sql_mapa)

### (c) Pirâmide etária dos óbitos

In [ ]:
sql_piramide = """
SELECT p.faixa_etaria, p.sexo, SUM(f.flag_obito_covid) AS obitos
FROM dw.fato_notificacao_covid f
JOIN dw.dim_perfil_paciente p ON p.sk_perfil = f.sk_perfil
GROUP BY p.faixa_etaria, p.sexo
ORDER BY p.faixa_etaria;
"""
print(sql_piramide)

### (d) Top-5 comorbidades em óbitos

In [ ]:
sql_comorbidades = """
SELECT
    CASE
        WHEN c.com_cardio   = 'Sim' THEN 'Cardiopatia'
        WHEN c.com_diabetes  = 'Sim' THEN 'Diabetes'
        WHEN c.com_obesidade = 'Sim' THEN 'Obesidade'
        WHEN c.com_pulmao   = 'Sim' THEN 'Doença Pulmonar'
        WHEN c.com_renal    = 'Sim' THEN 'Doença Renal'
    END AS comorbidade,
    SUM(f.flag_obito_covid) AS obitos
FROM dw.fato_notificacao_covid f
JOIN dw.dim_comorbidade c ON c.sk_como = f.sk_como
GROUP BY comorbidade
HAVING comorbidade IS NOT NULL
ORDER BY obitos DESC
LIMIT 5;
"""
print(sql_comorbidades)

---
## Exercício 6 — SCD Tipo 2 para DIM_LOCALIDADE

Caso a coluna `PopulacaoMunicipio` passe a variar no tempo, a dimensão deve rastrear versões históricas com as colunas de controle `data_inicio`, `data_fim` e `flag_atual`.

In [ ]:
sql_scd2_ddl = """
CREATE TABLE dw.dim_localidade_scd2 (
    sk_local            SERIAL PRIMARY KEY,
    municipio           VARCHAR(100),
    bairro              VARCHAR(150),
    uf                  CHAR(2)  DEFAULT 'ES',
    regiao_es           VARCHAR(30),
    macrorregiao        VARCHAR(30),
    populacao_munic     INT,          -- atributo que muda no tempo

    -- Controle SCD Tipo 2
    data_inicio         DATE NOT NULL,
    data_fim            DATE,         -- NULL = registro atual
    flag_atual          BOOLEAN NOT NULL DEFAULT TRUE,
    nk_municipio_bairro VARCHAR(260)  -- chave natural (não muda)
);
"""
print(sql_scd2_ddl)

In [ ]:
sql_scd2_update = """
-- 1. Fechar o registro vigente
UPDATE dw.dim_localidade_scd2
SET data_fim   = CURRENT_DATE - 1,
    flag_atual = FALSE
WHERE nk_municipio_bairro = 'Vitória|Centro'
  AND flag_atual = TRUE;

-- 2. Inserir nova versão
INSERT INTO dw.dim_localidade_scd2
    (municipio, bairro, uf, regiao_es, macrorregiao,
     populacao_munic, data_inicio, data_fim, flag_atual, nk_municipio_bairro)
VALUES
    ('Vitória', 'Centro', 'ES', 'Metropolitana', 'Grande Vitória',
     365000, CURRENT_DATE, NULL, TRUE, 'Vitória|Centro');
"""
print(sql_scd2_update)

A tabela fato continua apontando para o `sk_local` histórico correto — análises passadas permanecem íntegras. Análises atuais usam `WHERE flag_atual = TRUE` para obter a versão vigente.

---
## Exercício 7 — Stored Procedure de Validação de Qualidade

A procedure valida três condições antes de cada carga da fato:
- **(a)** existência do membro 'Desconhecido' (SK = -1) em toda dimensão
- **(b)** ausência de FKs órfãs na fato
- **(c)** contagem da fato bate com a staging

In [ ]:
sql_procedure = """
CREATE OR REPLACE PROCEDURE dw.validar_carga()
LANGUAGE plpgsql AS $$
DECLARE
    v_erro TEXT := '';
    v_count INT;
BEGIN
    -- (a) Verificar membro "Desconhecido" em todas as dimensões
    SELECT COUNT(*) INTO v_count FROM dw.dim_localidade     WHERE sk_local = -1;
    IF v_count = 0 THEN v_erro := v_erro || '[ERRO] dim_localidade sem membro -1. '; END IF;

    SELECT COUNT(*) INTO v_count FROM dw.dim_perfil_paciente WHERE sk_perfil = -1;
    IF v_count = 0 THEN v_erro := v_erro || '[ERRO] dim_perfil sem membro -1. '; END IF;

    SELECT COUNT(*) INTO v_count FROM dw.dim_classificacao   WHERE sk_class = -1;
    IF v_count = 0 THEN v_erro := v_erro || '[ERRO] dim_classificacao sem membro -1. '; END IF;

    SELECT COUNT(*) INTO v_count FROM dw.dim_sintomas         WHERE sk_sint = -1;
    IF v_count = 0 THEN v_erro := v_erro || '[ERRO] dim_sintomas sem membro -1. '; END IF;

    SELECT COUNT(*) INTO v_count FROM dw.dim_comorbidade      WHERE sk_como = -1;
    IF v_count = 0 THEN v_erro := v_erro || '[ERRO] dim_comorbidade sem membro -1. '; END IF;

    SELECT COUNT(*) INTO v_count FROM dw.dim_teste            WHERE sk_teste = -1;
    IF v_count = 0 THEN v_erro := v_erro || '[ERRO] dim_teste sem membro -1. '; END IF;

    SELECT COUNT(*) INTO v_count FROM dw.dim_tempo            WHERE sk_tempo = -1;
    IF v_count = 0 THEN v_erro := v_erro || '[ERRO] dim_tempo sem membro -1. '; END IF;

    -- (b) Verificar FKs órfãs
    SELECT COUNT(*) INTO v_count
    FROM dw.fato_notificacao_covid f
    WHERE NOT EXISTS (
        SELECT 1 FROM dw.dim_localidade d WHERE d.sk_local = f.sk_local
    );
    IF v_count > 0 THEN
        v_erro := v_erro || FORMAT('[ERRO] %s FKs orfas em sk_local. ', v_count);
    END IF;

    SELECT COUNT(*) INTO v_count
    FROM dw.fato_notificacao_covid f
    WHERE NOT EXISTS (
        SELECT 1 FROM dw.dim_tempo d WHERE d.sk_tempo = f.sk_data_notificacao
    );
    IF v_count > 0 THEN
        v_erro := v_erro || FORMAT('[ERRO] %s FKs orfas em sk_data_notificacao. ', v_count);
    END IF;

    -- (c) Contagem fato = staging
    DECLARE
        v_stg  BIGINT;
        v_fato BIGINT;
    BEGIN
        SELECT COUNT(*) INTO v_stg  FROM stg.notificacao_raw;
        SELECT COUNT(*) INTO v_fato FROM dw.fato_notificacao_covid;
        IF v_stg <> v_fato THEN
            v_erro := v_erro || FORMAT(
                '[ERRO] Contagem divergente: staging=%s, fato=%s. ', v_stg, v_fato
            );
        END IF;
    END;

    -- Resultado final
    IF v_erro = '' THEN
        RAISE NOTICE '[OK] Todas as validacoes passaram.';
    ELSE
        RAISE EXCEPTION 'Falhas na validacao: %', v_erro;
    END IF;
END;
$$;

-- Execução
CALL dw.validar_carga();
"""
print(sql_procedure)

---
*FAESA – Ciência da Computação | Business Intelligence 2026/1 | Prof. Otávio Lube*